# Wildfire Incident Command - GRPO Training (v2)

GRPO reinforcement learning on a wildfire incident command model, starting from the SFT checkpoint.

**Five critical issues fixed in this version:**
1. Prompt/reward state mismatch - dataset uses step-0 prompts only; reward replays the exact (tier, seed)
2. Truncated rollout - reward runs full episode to completion (heuristic continuation), terminal reward always included
3. Wasted inner model generations - MODEL_STEPS=1, only the sampled completion is applied
4. GRPO loop too slow - consequence of fix 3
5. parse_action(text, None) crash - standalone check_json_format() for format reward

**Hardware:** A100 Large 40GB (HuggingFace Space JupyterLab) — ~75 min wall-clock for 150 steps

**Before running:** In a terminal, authenticate:
```
huggingface-cli login   # HF token with write access (to load SFT model + push result)
wandb login             # wandb API key (Section 9 logs to wandb)
```
Also ensure the repo is cloned and this notebook is opened from inside the repo root (so `REPO_ROOT = "."` resolves correctly).

## Section 1 - Install and Assert GPU

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install "trl==0.20.0" datasets==3.4.1 wandb
# torchvision: choose the index matching your CUDA version
# HF Space A100/A10G (CUDA 12.8): use cu128
# Standard Colab (CUDA 12.1): replace cu128 with cu121
!pip install torchvision --index-url https://download.pytorch.org/whl/cu128

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU not available - switch to a GPU runtime"
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name}  |  VRAM: {gpu_mem:.1f} GB")

In [ ]:
import sys
from enum import Enum
import importlib.machinery
from unittest.mock import MagicMock

# torchvision C extension is ABI-incompatible with torch 2.10.0+cu128.
# Stub it out — text-only GRPO never calls vision ops.
# If your torchvision imports correctly, this cell is harmless (setdefault won't overwrite).
for _key in list(sys.modules.keys()):
    if 'torchvision' in _key:
        del sys.modules[_key]

class _InterpolationMode(Enum):
    NEAREST = "nearest"
    NEAREST_EXACT = "nearest_exact"
    BOX = "box"
    BILINEAR = "bilinear"
    BICUBIC = "bicubic"
    HAMMING = "hamming"
    LANCZOS = "lanczos"

class _StubModule(type(sys)): 
    def __getattr__(self, name):
        if name.startswith('__'):
            raise AttributeError(name)
        mock = MagicMock()
        setattr(self, name, mock)
        return mock

def _make(name):
    m = _StubModule(name)
    m.__spec__    = importlib.machinery.ModuleSpec(name, None)
    m.__path__    = []
    m.__package__ = name
    return m

_tv      = _make("torchvision")
_tv.__version__ = "0.20.0"
_tr      = _make("torchvision.transforms")
_tr.InterpolationMode = _InterpolationMode
_tr_v2   = _make("torchvision.transforms.v2")
_tvF     = _make("torchvision.transforms.v2.functional")
_ops     = _make("torchvision.ops")
_models  = _make("torchvision.models")
_io      = _make("torchvision.io")
_utils   = _make("torchvision.utils")
_datasets= _make("torchvision.datasets")
_tv.transforms = _tr
_tr.v2 = _tr_v2
_tr_v2.functional = _tvF
_tv.ops = _ops; _tv.models = _models; _tv.io = _io
_tv.utils = _utils; _tv.datasets = _datasets

for _mod in [_tv, _tr, _tr_v2, _tvF, _ops, _models, _io, _utils, _datasets]:
    sys.modules[_mod.__name__] = _mod

print("torchvision stubbed OK (safe for text-only training)")

## Section 2 - Load SFT Checkpoint

In [ ]:
from unsloth import FastLanguageModel

# Option A: Load from HuggingFace Hub
SFT_MODEL = "Eshit/wildfire-sft-7b"
# Option B: Load from local zip (uncomment and adjust if needed)
# !unzip sft_final.zip -d sft_final_dir
# SFT_MODEL = "./sft_final_dir/sft_final"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_MODEL,
    max_seq_length=2048,
    load_in_4bit=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loaded SFT checkpoint: {SFT_MODEL}")
model.print_trainable_parameters()

## Section 3 - Constants and Controller Setup

In [ ]:
import os, random, json, sys
import torch

# Clone the simulator repo first (run once in a terminal or notebook cell):
# !git clone https://github.com/Abrodolph/Wildfire-Containment-Simulator /home/user/app/Wildfire-Containment-Simulator
# !pip install -e /home/user/app/Wildfire-Containment-Simulator --quiet
REPO_ROOT = "/home/user/app/Wildfire-Containment-Simulator"  # HF JupyterLab path
# On standard Colab: REPO_ROOT = "/content/Wildfire-Containment-Simulator"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from env.wildfire_env import WildfireEnv
from env.serialization import serialize_observation
from env.action_parser import parse_action
from agents.heuristic_agent import HeuristicAgent
from env.curriculum import CurriculumController
from datasets import Dataset

SEED_POOL = list(range(100))
TIER_MAX_STEPS = {'easy': 80, 'medium': 150, 'hard': 300}
SYSTEM_PROMPT = (
    'You are an AI Incident Commander managing wildfire containment. '
    'You will receive a situation briefing each step. '
    'Respond with ONLY a valid JSON action object and nothing else. '
    'Example: {"action_type": "idle"}'
)

# Thresholds calibrated to full-episode reward with heuristic continuation.
# Promote easy->medium once model's first action consistently beats random (+6.23).
# Promote medium->hard once model demonstrates meaningful improvement over random (+1.31).
controller = CurriculumController(
    start_tier='easy',
    thresholds={'easy': 6.5, 'medium': 3.5},
)

os.makedirs('training/samples', exist_ok=True)
_reward_call_count = 0

print(f"Start tier: {controller.get_tier()}")
print(f"Seed pool: {len(SEED_POOL)} seeds")
print("Env imports OK")

## Section 4 - Standalone JSON Format Checker

Replaces parse_action for format reward - no obs object needed (Issue 5 fix).

In [ ]:
import json as _json
import re as _re
from env.models import ActionType as _AT

_VALID_ACTION_TYPES = {a.value for a in _AT}


def check_json_format(text: str) -> str:
    """
    Validate LLM output format without needing an obs object.
    Returns 'json_success', 'regex_fallback', or 'safe_idle'.
    Does NOT use parse_action - avoids the obs.grid dependency.
    """
    text = _re.sub(r'```(?:json)?\s*', '', text).replace('```', '')
    start = text.find('{')
    if start == -1:
        return 'safe_idle'
    depth = 0
    end = -1
    for i, ch in enumerate(text[start:], start=start):
        if ch == '{':
            depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0:
                end = i
                break
    if end == -1:
        return 'safe_idle'
    try:
        obj = _json.loads(text[start:end+1])
        if not isinstance(obj, dict):
            return 'safe_idle'
        at = str(obj.get('action_type', '')).lower()
        if at in _VALID_ACTION_TYPES:
            return 'json_success'
        return 'regex_fallback'
    except Exception:
        return 'regex_fallback'


assert check_json_format('{"action_type": "idle"}') == 'json_success'
assert check_json_format('{"action_type": "bogus"}') == 'regex_fallback'
assert check_json_format('no json here') == 'safe_idle'
print('check_json_format OK')

## Section 5 - Reward Functions

Two reward signals for GRPO:
- **reward_fn_outcome** - full-episode env reward (1 model step + heuristic continuation)
- **reward_fn_format** - JSON formatting quality (fast, no env needed)

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def _run_episode(args):
    """Run one full wildfire episode for a single GRPO completion (parallelizable)."""
    completion, ep_tier, ep_seed = args
    env = WildfireEnv()
    obs = env.reset(task_id=ep_tier, seed=ep_seed)
    total_reward = 0.0
    text = completion if isinstance(completion, str) else completion[0]['content']
    action, _ = parse_action(text, obs)
    result = env.step(action)
    total_reward += result.reward
    obs = result.observation
    heuristic = HeuristicAgent()
    while not env.done:
        action = heuristic.act(obs)
        result = env.step(action)
        total_reward += result.reward
        obs = result.observation
    return total_reward


def reward_fn_outcome(completions, prompts, tier=None, seed=None, **kwargs):
    """
    Score each GRPO completion by:
      1. Resetting the env to the EXACT (tier, seed) that generated the prompt (Issue 1 fix).
      2. Applying the sampled completion as the single first action (MODEL_STEPS=1, Issue 3/4 fix).
      3. Running HeuristicAgent until episode completion (Issue 2 fix - captures terminal reward).
    Episodes are run in parallel threads to reduce wall-clock time.
    tier and seed are dataset columns forwarded by GRPOTrainer.
    """
    global _reward_call_count
    _reward_call_count += 1

    args_list = [
        (
            completions[i],
            tier[i] if tier is not None else controller.get_tier(),
            seed[i] if seed is not None else random.choice(SEED_POOL),
        )
        for i in range(len(completions))
    ]

    with ThreadPoolExecutor(max_workers=len(completions)) as executor:
        rewards = list(executor.map(_run_episode, args_list))

    mean_r = sum(rewards) / len(rewards)
    promoted = controller.after_episode(mean_r)
    if promoted:
        print(f'  *** Curriculum promoted to: {promoted} (mean batch reward={mean_r:.2f}) ***')

    if _reward_call_count % 10 == 0:
        os.makedirs('training/samples', exist_ok=True)
        sample_path = f'training/samples/call_{_reward_call_count}.txt'
        with open(sample_path, 'w') as f:
            f.write(f'call={_reward_call_count}  tier={tier[0] if tier else "?"}  reward={rewards[0]:.3f}\n')
            f.write('---\n')
            c = completions[0]
            f.write(c if isinstance(c, str) else c[0]['content'])

    return rewards


def reward_fn_format(completions, prompts, **kwargs):
    """
    Scores JSON formatting quality using check_json_format() (no obs needed).
    Runs independently of the env - fast and always well-defined.
    """
    rewards = []
    for completion in completions:
        text = completion if isinstance(completion, str) else completion[0]['content']
        status = check_json_format(text)
        if status == 'json_success':
            r = 0.15
        elif status == 'regex_fallback':
            r = 0.0
        else:
            r = -0.20
        rewards.append(r)
    return rewards


print('Reward functions defined (parallelized).')

## Section 6 - Dataset Builder (Step-0 Only)

Each row stores the seed so reward_fn_outcome can replay the exact same env state.
No mid-episode offset - GRPO prompt and reward state are always step-0.

In [ ]:
def build_prompt_dataset(n=200):
    """
    Build step-0 prompts for the current curriculum tier.
    Stores the seed in each row so reward_fn can replay the exact same env state.
    """
    rows = []
    env_tmp = WildfireEnv()
    tier = controller.get_tier()
    max_steps = TIER_MAX_STEPS[tier]

    for i in range(n):
        seed = SEED_POOL[i % len(SEED_POOL)]
        obs = env_tmp.reset(task_id=tier, seed=seed)
        prompt = serialize_observation(obs, 0, max_steps, tier=tier, prev_cells_burning=0)
        rows.append({
            'prompt': [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user',   'content': prompt},
            ],
            'tier': tier,
            'seed': seed,
        })
    return rows


_test_ds = build_prompt_dataset(3)
print(f"Sample dataset row keys: {list(_test_ds[0].keys())}")
print(f"Tier: {_test_ds[0]['tier']}, Seed: {_test_ds[0]['seed']}")
print(f"Prompt roles: {[m['role'] for m in _test_ds[0]['prompt']]}")
del _test_ds

## Section 7 - CurriculumDatasetCallback

Rebuilds the training dataset whenever the curriculum controller promotes to a new tier.

In [ ]:
from transformers import TrainerCallback


class CurriculumDatasetCallback(TrainerCallback):
    def __init__(self, trainer_ref):
        self._trainer = trainer_ref
        self._last_tier = controller.get_tier()

    def on_step_end(self, args, state, control, **kwargs):
        current_tier = controller.get_tier()
        if current_tier != self._last_tier:
            print(f'  Rebuilding dataset for tier: {current_tier}')
            new_ds = Dataset.from_list(build_prompt_dataset(200))
            self._trainer.train_dataset = new_ds
            self._last_tier = current_tier


print('CurriculumDatasetCallback defined.')

## Section 8 - GRPOTrainer Setup

In [ ]:
from trl import GRPOTrainer, GRPOConfig

grpo_config = GRPOConfig(
    output_dir='./grpo_checkpoints',
    num_generations=8,
    learning_rate=3e-6,
    max_steps=150,       # 150 steps ~ 75 min on A100; increase to 400 if time allows
    save_steps=10,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_completion_length=192,
    logging_steps=1,
    report_to='wandb',
)

FastLanguageModel.for_training(model)

dataset = Dataset.from_list(build_prompt_dataset(200))
print(f'Initial dataset: {len(dataset)} rows, tier={controller.get_tier()}')

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_fn_outcome, reward_fn_format],
    args=grpo_config,
    train_dataset=dataset,
)
trainer.add_callback(CurriculumDatasetCallback(trainer))

print('GRPOTrainer ready.')

## Section 9 - Run Training

In [ ]:
import wandb
wandb.init(project='wildfire-grpo', name='qwen7b-v2')

print(f'Starting GRPO - {grpo_config.max_steps} steps, {grpo_config.num_generations} gen/prompt')
print(f'Reward: 1 model step at step-0, heuristic continuation to episode completion')
print(f'Start tier: {controller.get_tier()}')

trainer.train()
print('Training complete.')

history = controller.get_history()
stats = [{'step': ep, 'tier': t, 'mean_reward': r} for ep, t, r in history]
with open('./training_stats.json', 'w') as f:
    json.dump(stats, f, indent=2)
print('Stats saved -> training_stats.json')

# To resume training for more steps later:
# grpo_config.max_steps = 300  # new total
# trainer.train(resume_from_checkpoint='./grpo_checkpoints')

## Section 10 - Evaluate vs Baselines

Run 15 full episodes per tier (seeds 42-56), compare with heuristic and random baselines.

In [ ]:
class LLMAgent:
    """Wraps the trained model for evaluation. Must be re-instantiated per episode."""

    def __init__(self, model, tokenizer, tier, max_steps):
        self.model = model
        self.tokenizer = tokenizer
        self.tier = tier
        self.max_steps = max_steps
        self._step = 0
        self._prev_burning = 0
        self.json_success = self.regex_fallback = self.safe_idle = 0

    def act(self, obs):
        prompt = serialize_observation(
            obs, self._step, self.max_steps,
            tier=self.tier,
            prev_cells_burning=self._prev_burning,
        )
        self._prev_burning = obs.stats.cells_burning
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': prompt},
        ]
        input_ids = self.tokenizer.apply_chat_template(
            messages, tokenize=True,
            add_generation_prompt=True, return_tensors='pt',
        ).to(self.model.device)
        with torch.no_grad():
            out = self.model.generate(
                input_ids, max_new_tokens=128,
                pad_token_id=self.tokenizer.eos_token_id,
            )
        text = self.tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)
        action, status = parse_action(text, obs)
        if status == 'json_success':
            self.json_success += 1
        elif status == 'regex_fallback':
            self.regex_fallback += 1
        else:
            self.safe_idle += 1
        self._step += 1
        return action


print('LLMAgent class defined.')

In [ ]:
import numpy as np

# Adjust path to repo root if needed
BASELINES_PATH = f'{REPO_ROOT}/scripts/results.json'
with open(BASELINES_PATH, 'r') as f:
    baselines = json.load(f)

FastLanguageModel.for_inference(model)

EVAL_SEEDS = list(range(42, 57))
TIERS = ['easy', 'medium', 'hard']

results = {}

for tier in TIERS:
    max_steps = TIER_MAX_STEPS[tier]
    tier_rewards = []
    tier_pop_saved = []
    tier_json_success = 0
    tier_total_actions = 0

    print(f'\nEvaluating {tier} tier...')

    for seed in EVAL_SEEDS:
        agent = LLMAgent(model, tokenizer, tier, max_steps)
        env = WildfireEnv()
        obs = env.reset(task_id=tier, seed=seed)
        total_reward = 0.0

        while not env.done:
            action = agent.act(obs)
            result = env.step(action)
            total_reward += result.reward
            obs = result.observation

        tier_rewards.append(total_reward)

        state = env.state()
        total_pop = state['total_population']
        pop_lost = state['population_lost']
        pop_saved = 100.0 * (total_pop - pop_lost) / total_pop if total_pop > 0 else 100.0
        tier_pop_saved.append(pop_saved)

        tier_json_success += agent.json_success
        tier_total_actions += agent.json_success + agent.regex_fallback + agent.safe_idle

        print(f'  seed={seed}: reward={total_reward:+.2f}, pop_saved={pop_saved:.0f}%')

    json_rate = 100.0 * tier_json_success / tier_total_actions if tier_total_actions > 0 else 0
    results[tier] = {
        'mean': float(np.mean(tier_rewards)),
        'std': float(np.std(tier_rewards)),
        'pop_saved_pct': float(np.mean(tier_pop_saved)),
        'json_success_rate': json_rate,
    }

print()
print('=' * 65)
print('=== Evaluation: Trained Model vs Baselines ===')
print('Seeds: 42-56  (15 per tier)')
print('=' * 65)
header = f'{"Tier":<10} {"Trained":>12} {"Heuristic":>12} {"Random":>12} {"vs Heur.":>12}'
print(header)
print('-' * 65)

any_tier_close = False
for tier in TIERS:
    t = results[tier]
    h_mean = baselines['heuristic'][tier]['mean']
    h_std = baselines['heuristic'][tier]['std']
    r_mean = baselines['random'][tier]['mean']
    r_std = baselines['random'][tier]['std']
    delta = t['mean'] - h_mean
    marker = ' OK' if delta >= -1.0 else ''
    if delta >= -1.0:
        any_tier_close = True
    print(
        f'{tier:<10} '
        f'{t["mean"]:+.2f}+/-{t["std"]:.1f}  '
        f'{h_mean:+.2f}+/-{h_std:.1f}  '
        f'{r_mean:+.2f}+/-{r_std:.1f}  '
        f'{delta:+.2f}{marker}'
    )

print()
print('JSON success rate:  ', end='')
print('  '.join(f'{t}={results[t]["json_success_rate"]:.1f}%' for t in TIERS))
print('Pop saved rate:     ', end='')
print('  '.join(f'{t}={results[t]["pop_saved_pct"]:.0f}%' for t in TIERS))

with open('./grpo_eval_results.json', 'w') as f:
    json.dump({
        'trained': results,
        'baselines': baselines,
        'eval_seeds': EVAL_SEEDS,
        'model': 'Eshit/wildfire-grpo-7b',
    }, f, indent=2)
print('Eval results saved -> grpo_eval_results.json')

assert any_tier_close, (
    'Trained model did not come within 1.0 of heuristic on any tier. '
    'Check training logs and sample completions.'
)
print('\nPASS: At least one tier within 1.0 of heuristic baseline.')

FastLanguageModel.for_training(model)

## Section 11 - Save and Push

In [ ]:
model.save_pretrained('./grpo_final')
tokenizer.save_pretrained('./grpo_final')
print('Saved to ./grpo_final')

In [ ]:
HF_USERNAME = 'Eshit'  # <-- CHANGE THIS
model.push_to_hub(f'{HF_USERNAME}/wildfire-grpo-7b')
tokenizer.push_to_hub(f'{HF_USERNAME}/wildfire-grpo-7b')
print(f'Pushed to hub: {HF_USERNAME}/wildfire-grpo-7b')

In [ ]:
!zip -r grpo_final.zip ./grpo_final
print('Zipped to grpo_final.zip')
# On HF JupyterLab: right-click grpo_final.zip in the file browser and choose Download.
# On Google Colab only (not needed here):
# from google.colab import files; files.download('grpo_final.zip')